<a href="https://colab.research.google.com/github/marina-popova11/MediaImpactOnCryptoPrices/blob/training/distil_bert/two_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader
from datasets import DatasetDict, Dataset, load_from_disk
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModel, DistilBertForSequenceClassification
from tqdm import tqdm

import wandb
wandb.init(mode="offline")

def load_arrow(filepath):
    try:
        dataset = load_from_disk(filepath)
        print("Successfully loaded dataset from disk")
        return dataset
    except Exception as e:
        print(e)
        return None

dataset_path = "/content/drive/MyDrive/date_with_all_move"

if os.path.exists(dataset_path):
    print(f"Path exists: {os.path.exists(dataset_path)}")
    print(f"Files in directory: {os.listdir(dataset_path)}")
else:
    print(f"Path does not exist. Current directory: {os.getcwd()}")
    print(f"Available files: {os.listdir('.')}")

dataset = load_arrow(dataset_path)
print(f"\nDataset type: {type(dataset)}")
print(f"Dataset keys: {list(dataset.keys())}")

model_path = "/content/drive/MyDrive/bert_final_81_percent"
text_encoder = AutoModel.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
text_encoder.eval()
text_encoder = text_encoder.to("cuda" if torch.cuda.is_available() else "cpu")

sentiment_model = DistilBertForSequenceClassification.from_pretrained(model_path)
sentiment_model.eval()
sentiment_model = sentiment_model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def extract_sentiment_scores(texts, batch_size=64):
    device = next(sentiment_model.parameters()).device
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Извлечение сентимента"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)
        with torch.no_grad():
            outputs = sentiment_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)  # shape (N, 3)
            scores.append(probs.cpu())
    return torch.cat(scores, dim=0)  # (N, 3)

In [ ]:
def extract_embeddings(texts, batch_size=64):
    device = next(text_encoder.parameters()).device
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Извлечение эмбеддингов"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)
        with torch.no_grad():
            outputs = text_encoder(**inputs)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_emb.cpu())
    return torch.cat(embeddings, dim=0)

In [ ]:
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
print(train_df.shape, val_df.shape)

train_emb = extract_embeddings(train_df["text"].tolist())
val_emb = extract_embeddings(val_df["text"].tolist())
test_emb = extract_embeddings(test_df["text"].tolist())
import torch.nn.functional as F

train_emb = F.normalize(train_emb, p=2, dim=1)
val_emb = F.normalize(val_emb, p=2, dim=1)
test_emb = F.normalize(test_emb, p=2, dim=1)

In [ ]:
train_sent = extract_sentiment_scores(train_df["text"].tolist())
val_sent = extract_sentiment_scores(val_df["text"].tolist())
test_sent = extract_sentiment_scores(test_df["text"].tolist())

numeric_cols = ["price_at_t", "price_1h_before", "Volume"]

scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[numeric_cols])
X_val_num = scaler.transform(val_df[numeric_cols])
X_test_num = scaler.transform(test_df[numeric_cols])

X_train_num = torch.tensor(X_train_num, dtype=torch.float32)
X_val_num = torch.tensor(X_val_num, dtype=torch.float32)
X_test_num = torch.tensor(X_test_num, dtype=torch.float32)

X_train = torch.cat([train_emb, X_train_num, train_sent], dim=1)
X_val = torch.cat([val_emb, X_val_num, val_sent], dim=1)
X_test = torch.cat([test_emb, X_test_num, test_sent], dim=1)

y_train = torch.tensor(train_df["move_6"].values, dtype=torch.long)
y_val = torch.tensor(val_df["move_6"].values, dtype=torch.long)
y_test = torch.tensor(test_df["move_6"].values, dtype=torch.long)

In [ ]:
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train.numpy()),
    y=y_train.numpy()
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)
print("Train class counts:", np.bincount(y_train))

In [ ]:
import torch

class MLPHead(nn.Module):
    def __init__(self, input_dim=774, dropout=0.3, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_head = MLPHead(input_dim=774).to(device)
optimizer = torch.optim.AdamW(model_head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
from torch.utils.data import WeightedRandomSampler

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, pin_memory=True, num_workers=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128)

best_f1 = 0
patience = 5
patience_counter = 0
for epoch in range(20):
    model_head.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model_head(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model_head.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model_head(X_batch)
            pred = logits.argmax(dim=1).cpu()
            preds.extend(pred.tolist())
            targets.extend(y_batch.cpu().tolist())

    val_f1 = f1_score(targets, preds)
    print(f"Epoch {epoch+1}, Train Loss: {total_loss/len(train_loader):.4f}, Val F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model_head.state_dict(), "best_mlp_move6.pth")
        print(f"  → Новое лучшее значение F1: {best_f1:.4f}, модель сохранена.")
    else:
        patience_counter += 1
        print(f"  → Нет улучшения. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\n[Early stopping] Лучший F1: {best_f1:.4f}")
        break
    if val_f1 > best_f1:
        best_f1 = val_f1

In [ ]:
model_head.load_state_dict(torch.load("best_mlp_move6.pth"))
model_head.eval()

val_preds = []
val_probs, val_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        logits = model_head(X_batch)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu()
        pred = logits.argmax(dim=1).cpu()
        val_preds.extend(pred.tolist())
        val_probs.extend(probs.tolist())
        val_targets.extend(y_batch.tolist())

print("Val Predicted class distribution:", np.bincount(val_preds, minlength=2))
print("Val True class distribution:      ", np.bincount(val_targets, minlength=2))

best_thresh = 0.5
best_metric = 0.0
for thresh in np.arange(0.2, 0.6, 0.01):
    preds = (np.array(val_probs) >= thresh).astype(int)
    f1 = f1_score(val_targets, preds)
    if f1 > best_metric:
        best_metric = f1
        best_thresh = thresh

print(f"Лучший порог: {best_thresh:.2f}, F1 на валидации: {best_metric:.4f}")

In [ ]:
from sklearn.metrics import balanced_accuracy_score
model_head.load_state_dict(torch.load("best_mlp_move6.pth"))
model_head.eval()

test_probs, test_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model_head(X_batch)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu()
        test_probs.extend(probs.tolist())
        test_targets.extend(y_batch.tolist())

test_preds = (np.array(test_probs) >= best_thresh).astype(int)

print("\n=== Тестовые метрики ===")
print(classification_report(test_targets, test_preds, target_names=["No Move", "Move"]))
test_acc = np.mean(np.array(test_preds) == np.array(test_targets))
print(f"Accuracy: {test_acc:.4f}")
print(f"F1: {f1_score(test_targets, test_preds):.4f}")
print(f"Precision: {precision_score(test_targets, test_preds):.4f}")
print(f"Recall: {recall_score(test_targets, test_preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(test_targets, test_preds):.4f}")

In [ ]:
import joblib
joblib.dump(scaler, "scaler_move6.pkl")
tokenizer.save_pretrained("tokenizer_move6_1")
print("Токенизатор сохранён")
with open("best_threshold_move6.txt", "w") as f:
    f.write(str(best_thresh))
print(f"Порог сохранён: {best_thresh:.2f}")

In [ ]:
def filter_data(df):
    df = df[df["move_6"] == 1].copy()
    df = df[df["label_class_6"] != 0].copy()

    df["direction_label"] = (df["label_class_6"] == 1).astype(int)
    print(df["direction_label"].value_counts())
    return df

train_dir_df = filter_data(dataset['train'].to_pandas())
val_dir_df = filter_data(dataset['validation'].to_pandas())
test_dir_df = filter_data(dataset['test'].to_pandas())

In [ ]:
train_dir_emb = extract_embeddings(train_dir_df["text"].tolist())
val_dir_emb = extract_embeddings(val_dir_df["text"].tolist())
test_dir_emb = extract_embeddings(test_dir_df["text"].tolist())
import torch.nn.functional as F

train_dir_emb = F.normalize(train_dir_emb, p=2, dim=1)
val_dir_emb = F.normalize(val_dir_emb, p=2, dim=1)
test_dir_emb = F.normalize(test_dir_emb, p=2, dim=1)

train_dir_sent = extract_sentiment_scores(train_dir_df["text"].tolist())
val_dir_sent = extract_sentiment_scores(val_dir_df["text"].tolist())
test_dir_sent = extract_sentiment_scores(test_dir_df["text"].tolist())

def process_texts(texts):
    embeddings = extract_embeddings(texts)
    embeddings = F.normalize(embeddings, p=2, dim=1)
    sentiment = extract_sentiment_scores(texts)
    return embeddings, sentiment

train_dir_emb, train_dir_sent = process_texts(train_dir_df["text"].tolist())
val_dir_emb, val_dir_sent     = process_texts(val_dir_df["text"].tolist())
test_dir_emb, test_dir_sent   = process_texts(test_dir_df["text"].tolist())

X_train = torch.cat([train_dir_emb, train_dir_sent], dim=1)
X_val = torch.cat([val_dir_emb, val_dir_sent], dim=1)
X_test = torch.cat([test_dir_emb, test_dir_sent], dim=1)

y_train = torch.tensor(train_dir_df["direction_label"].values, dtype=torch.long)
y_val = torch.tensor(val_dir_df["direction_label"].values, dtype=torch.long)
y_test = torch.tensor(test_dir_df["direction_label"].values, dtype=torch.long)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train.numpy()
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

        if alpha is not None:
            self.alpha = torch.tensor(alpha, dtype=torch.float)

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)

        if self.alpha is not None:
            alpha_t = self.alpha[targets].to(inputs.device)
            loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            loss = (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

class DirectionMLP(nn.Module):
    def __init__(self, input_dim=771):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model_dir = DirectionMLP().to(device)
optimizer = torch.optim.AdamW(model_dir.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = FocalLoss(alpha=class_weights, gamma=2.0, reduction='mean')

In [ ]:
os.makedirs("direction_model", exist_ok=True)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

patience = 5
patience_counter = 0
best_f1 = 0
for epoch in range(20):
    model_dir.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model_dir(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    model_dir.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            logits = model_dir(X_batch)
            probs = torch.softmax(logits, dim=1)
            pred = (probs[:, 1] >= 0.5).int().cpu()
            preds.extend(pred.tolist())
            targets.extend(y_batch.tolist())

    val_f1 = f1_score(targets, preds, average="macro")
    print(f"Epoch {epoch+1}, Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model_dir.state_dict(), "direction_model/best_direction.pth")
        print(f"  → Новое лучшее значение F1: {best_f1:.4f}, модель сохранена.")
    else:
        patience_counter += 1
        print(f"  → Нет улучшения. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print(f"\n[Early stopping] Лучший F1: {best_f1:.4f}")
        break
    if val_f1 > best_f1:
        best_f1 = val_f1

print(f"Лучший F1 направления: {best_f1:.4f}")

In [ ]:
model_dir.load_state_dict(torch.load("direction_model/best_direction.pth"))
model_dir.eval()
val_probs = []
val_preds, val_targets = [], []
with torch.no_grad():
    for i in range(0, len(X_val), 128):
        X_batch = X_val[i:i+128].to(device)
        logits = model_dir(X_batch)
        probs = torch.softmax(logits, dim=1)  # (N, 2)
        val_probs.append(probs.cpu())
val_probs = torch.cat(val_probs, dim=0)
val_preds = (val_probs[:, 1] >= 0.5).int()
val_targets = y_val

print("Val Predicted class distribution:", np.bincount(val_preds.numpy(), minlength=2))
print("Val True class distribution:      ", np.bincount(val_targets.numpy(), minlength=2))
print("Val F1 (macro):", f1_score(val_targets, val_preds, average="macro"))

best_thresh, best_f1 = 0.5, 0
MIN_RECALL_UP = 0.35
for thresh in np.arange(0.4, 0.6, 0.01):
    preds = (val_probs[:, 1] >= thresh).int()
    rep = classification_report(y_val, preds, output_dict=True)
    recall_up = rep["1"]["recall"]
    macro_f1 = rep["macro avg"]["f1-score"]
    if recall_up >= MIN_RECALL_UP and macro_f1 > best_f1:
        best_macro_f1 = macro_f1
        best_thresh = thresh

print(f"Лучший порог: {best_thresh}")
print(f"Macro F1: {best_macro_f1:.4f}")
print(f"Recall Up: {recall_up:.4f}")

In [ ]:
model_dir.load_state_dict(torch.load("direction_model/best_direction.pth"))
model_dir.eval()
threshold = best_thresh
preds, targets = [], []
with torch.no_grad():
    for i in range(0, len(X_test), 128):
        X_batch = X_test[i:i+128].to(device)
        logits = model_dir(X_batch)
        probs = torch.softmax(logits, dim=1)  # shape: (batch, 2)
        pred = (probs[:, 1] >= threshold).int().cpu()  # 0 = Down, 1 = Up
        preds.extend(pred.tolist())
        targets.extend(y_test[i:i+128].tolist())

print("\n=== Тест направления (с порогом =", threshold, ") ===")
print(classification_report(targets, preds, target_names=["Down", "Up"]))